# 04 · Training

The trained avatar, its textured mesh with the Gaussians over it, on the frames kept out of training, next to the frames themselves. Run `avatar-splats train` first; this notebook only reads its results.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from avatar_splats.body_models.hybrid import Avatar
from avatar_splats.config import load_config
from avatar_splats.fitting import fitted_path
from avatar_splats.gaussians import BoundGaussians, GaussianRecord
from avatar_splats.training import (
    ColourCorrection,
    ColourRecord,
    Evaluation,
    Scene,
    TextureRecord,
    Textures,
    load_posing,
    load_views,
    target,
    trained_path,
)

config = load_config("../configs/default.yaml")
device = config.training.device
avatar = Avatar.load(fitted_path(config, "avatar"))
scene = Scene(
    avatar,
    Textures(TextureRecord.load(trained_path(config, "textures"))),
    BoundGaussians(GaussianRecord.load(trained_path(config, "gaussians"))),
).to(device)
colours = ColourRecord.load(trained_path(config, "colours"))
colours = ColourCorrection.from_record(colours).to(device)
evaluation = Evaluation.load(trained_path(config, "evaluation"))
views = load_views(config)
posing = load_posing(config, avatar)
print(f"{len(scene.gaussians)} Gaussians")

## Quality

PSNR in dB where the performer is or the avatar draws something, never on empty background (and above the chin for the head videos). Training frames are rendered with their learned pose corrections; held-out frames, never trained on, as the fits posed them.

In [ ]:
for video in np.unique(evaluation.videos):
    for held_out, marker in ((False, "."), (True, "o")):
        chosen = (evaluation.videos == video) & (evaluation.held_out == held_out)
        split = "held out" if held_out else "training"
        label = f"{video}, {split}: {evaluation.psnr[chosen].mean():.1f} dB"
        plt.plot(evaluation.frames[chosen], evaluation.psnr[chosen], marker, label=label)
plt.xlabel("frame")
plt.ylabel("PSNR (dB)")
plt.legend();

## Consistency between views

How much each point on the body changes colour as the avatar is seen from all around, above and below eye level too (standard deviation across views, 0 to 1). The training frames cannot show this: they are the views the Gaussians were fitted to. The textured mesh alone, the same from every side, scores about 0.02 by this measure, the floor.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (part, spread) in zip(
    axes, (("body", evaluation.view_spread), ("head", evaluation.head_view_spread)), strict=True
):
    ax.hist(spread, bins=50)
    ax.set_xlabel("colour standard deviation across views")
    ax.set_ylabel("points")
    ax.set_title(f"{part}: median {np.median(spread):.3f}, p90 {np.quantile(spread, 0.9):.3f}")

## Textures

The textures, baked from the training frames as training corrected their poses, and their coverage: how squarely the best frame saw each texel (black: never). Gaussians grow only where coverage is high; elsewhere the texture shows alone, filled in from its surroundings where nothing was seen.

In [ ]:
record = scene.textures.record()
fig, axes = plt.subplots(2, 2, figsize=(16, 9), width_ratios=(1, 2.5))
for row, (texture, coverage) in enumerate(
    ((record.body, record.body_coverage), (record.head, record.head_coverage))
):
    axes[row, 0].imshow(texture)
    axes[row, 1].imshow(np.concatenate((texture, np.repeat(coverage[..., None], 3, -1)), axis=0))
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()

## Renders

Top: the frame with the background removed. Middle: the textured mesh alone. Bottom: the avatar, the Gaussians drawn over the mesh. Pick a video and frames below.

In [ ]:
video = "body"
chosen = [v for v in views if v.video == video and not v.frame % config.training.holdout_every][:5]
black = torch.zeros(3, device=device)
fig, axes = plt.subplots(3, len(chosen), figsize=(4 * len(chosen), 15))
with torch.no_grad():
    for column, view in enumerate(chosen):
        vertices = posing.vertices(view.video, view.index)
        mesh = scene.draw_mesh(vertices, view)
        colour, alpha = scene.render(
            vertices, view, config.training.sh_degree, config.training.occlusion_margin
        )
        axes[0, column].imshow(target(view, black).cpu().numpy())
        axes[1, column].imshow(colours(view.video, mesh.image, mesh.mask).clamp(0, 1).cpu().numpy())
        axes[2, column].imshow(colours(view.video, colour, alpha).clamp(0, 1).cpu().numpy())
        axes[0, column].set_title(f"frame {view.frame}")
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()